In [ ]:
!git clone https://github.com/spMohanty/PlantVillage-Dataset

In [ ]:
!git clone https://github.com/aldrin233/RiceDiseases-DataSet.git

In [ ]:
!pip install tensorflow_model_optimization

6 classes


In [ ]:
# ===============================================================
# SimCLR + MADA-lite + SE-ResNet + INT8 Quantization (6 Classes)
# ===============================================================

import os, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ================= CONFIG =================
IMG_SIZE = 48
BATCH_SIZE = 32
EPOCHS_DA = 20
TEMPERATURE = 0.1

FEATURE_DIM = 64       # reduced
PROJECTION_DIM = 64    # reduced
NUM_DOMAINS = 2

SOURCE_DIR = '/content/PlantVillage-Dataset/raw/segmented'
TARGET_DIR = '/content/RiceDiseases-DataSet'

SOURCE_CLASSES = [
    "Tomato___Bacterial_spot",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Potato___Early_blight",
    "Corn_(maize)___Cercospora_leaf_spot_Gray_leaf_spot",
    "Tomato___Septoria_leaf_spot",
    "Strawberry___Leaf_scorch"
]

TARGET_CLASSES = sorted([d for d in os.listdir(TARGET_DIR) if os.path.isdir(os.path.join(TARGET_DIR, d))])

# ================= GRADIENT REVERSAL =================
@tf.custom_gradient
def grad_reverse(x, lambd):
    def grad(dy):
        return -lambd * dy, None
    return x, grad

class GradientReversal(layers.Layer):
    def __init__(self):
        super().__init__()
        self.lambd = tf.Variable(0.0, trainable=False, dtype=tf.float32)
    def call(self, x):
        return grad_reverse(x, self.lambd)

# ================= AUGMENTATION =================
def strong_aug(x):
    x = tf.image.random_brightness(x, 0.5)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = tf.image.random_flip_left_right(x)
    return tf.clip_by_value(x, 0, 1)

def weak_aug(x):
    x = tf.image.random_brightness(x, 0.3)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    return tf.clip_by_value(x, 0, 1)

def create_views(img, is_source):
    img = tf.cast(img, tf.float32) / 255.0
    aug = strong_aug if is_source else weak_aug
    return aug(img), aug(img)

# ================= DATA GENERATOR =================
def combined_generator():
    datagen = ImageDataGenerator()
    s_gen = datagen.flow_from_directory(
        SOURCE_DIR, classes=SOURCE_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2, shuffle=True, class_mode=None
    )
    t_gen = datagen.flow_from_directory(
        TARGET_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2, shuffle=True, class_mode=None
    )
    while True:
        xs = next(s_gen)
        xt = next(t_gen)
        X = np.concatenate([xs, xt], axis=0)
        domain = tf.concat([tf.zeros(xs.shape[0], dtype=tf.int32),
                            tf.ones(xt.shape[0], dtype=tf.int32)], axis=0)
        v1, v2 = [], []
        for i, img in enumerate(X):
            a, b = create_views(img, i < xs.shape[0])
            v1.append(a)
            v2.append(b)
        yield (tf.stack(v1), tf.stack(v2)), domain

# ================= SE-RESNET BACKBONE =================
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1,1,filters))(se)
    return layers.multiply([x, se])

def residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = se_block(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_encoder():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(16, 3, padding='same')(inp)  # reduced filters
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    for f, s in zip([32, 32, 64, 64, 128, 128], [1,1,2,1,2,1]):  # reduced filters
        x = residual_block(x, f, s)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(FEATURE_DIM, activation='relu', name='shared_features')(x)
    return models.Model(inp, x, name="encoder_agronet")

def projection_head():
    inp = layers.Input((FEATURE_DIM,))
    x = layers.Dense(32, activation='relu')(inp)
    out = layers.Dense(PROJECTION_DIM)(x)
    return models.Model(inp, out, name="projection")

def domain_head(grl):
    inp = layers.Input((FEATURE_DIM,))
    x = grl(inp)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(NUM_DOMAINS, activation='softmax')(x)
    return models.Model(inp, out, name="domain")

# ================= NT-XENT LOSS =================
def nt_xent(z1, z2):
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)
    z = tf.concat([z1, z2], axis=0)
    sim = tf.matmul(z, z, transpose_b=True)
    sim -= tf.eye(tf.shape(z)[0]) * 1e9
    bs = tf.shape(z1)[0]
    labels = tf.concat([tf.range(bs, 2*bs), tf.range(bs)], axis=0)
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, sim / TEMPERATURE, from_logits=True)
    return tf.reduce_mean(loss)

# ================= SIMCLR + MADA-LITE MODEL =================
class SimCLR_DA(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = build_encoder()
        self.projector = projection_head()
        self.grl = GradientReversal()
        self.domain = domain_head(self.grl)

    def train_step(self, data):
        (v1, v2), dom = data
        with tf.GradientTape() as tape:
            h1 = self.encoder(v1, training=True)
            h2 = self.encoder(v2, training=True)
            z1 = self.projector(h1, training=True)
            z2 = self.projector(h2, training=True)
            loss_c = nt_xent(z1, z2)
            pseudo = tf.stop_gradient(tf.nn.softmax(tf.concat([h1,h2], axis=0)))
            weights = tf.reduce_max(pseudo, axis=1)
            dom_pred = self.domain(tf.concat([h1,h2], axis=0), training=True)
            loss_d = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    tf.concat([dom, dom], axis=0), dom_pred
                ) * weights
            )
            loss = loss_c + loss_d
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {"contrastive_loss": loss_c, "domain_loss": loss_d}

# ================= PHASE 1: DOMAIN ADAPTATION =================
model = SimCLR_DA()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4))
gen = combined_generator()
print("\n Training SimCLR + MADA-lite DA")
for epoch in range(EPOCHS_DA):
    model.grl.lambd.assign(min(1.0, epoch / 10.0))
    print(f"Epoch {epoch+1} | DOMAIN_LAMBDA = {model.grl.lambd.numpy():.2f}")
    model.fit(gen, steps_per_epoch=200, epochs=1, verbose=1)
model.encoder.save("encoder_da_agronet.h5")

# ================= PHASE 2: LINEAR EVALUATION =================
model.encoder.trainable = False
classifier = tf.keras.Sequential([
    layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    model.encoder,
    layers.Dense(len(TARGET_CLASSES), activation='softmax')
])
classifier.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_gen = datagen.flow_from_directory(
    TARGET_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32, subset='training', class_mode='categorical'
)
val_gen = datagen.flow_from_directory(
    TARGET_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32, subset='validation', class_mode='categorical', shuffle=False
)

print("\nTraining Linear Classifier")
classifier.fit(train_gen, epochs=20, validation_data=val_gen, verbose=1)
loss, acc = classifier.evaluate(val_gen, verbose=0)

# ================= RESULTS =================
classifier.save("full_da_model_agronet.h5")
encoder_size = os.path.getsize("encoder_da_agronet.h5") / 1024  # KB
full_size = os.path.getsize("full_da_model_agronet.h5") / 1024  # KB
print("\n==============================")
print(f"Target Accuracy : {acc*100:.2f}%")
print(f" Encoder Size   : {encoder_size:.2f} KB")
print(f" Full Model Size: {full_size:.2f} KB")
print("==============================")

# ================= TFLITE CONVERSION + INT8 QUANT =================
def representative_data_gen():
    for images, _ in train_gen:
        yield [images.astype(np.float32)]
        break  # only one batch is enough

tflite_model_file = "full_da_model_agronet_int8_small.tflite"
converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_model = converter.convert()
with open(tflite_model_file, "wb") as f:
    f.write(tflite_model)
tflite_size = os.path.getsize(tflite_model_file)  # bytes
print(f" INT8 TFLite Model Size: {tflite_size/1024:.2f} KB")

9 Classes

In [ ]:
# ===============================================================
# SimCLR + MADA-lite + SE-ResNet + INT8 Quantization (9 classes)
# ===============================================================

import os, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ================= CONFIG =================
IMG_SIZE = 48
BATCH_SIZE = 32
EPOCHS_DA = 20
TEMPERATURE = 0.1

FEATURE_DIM = 64       # reduced
PROJECTION_DIM = 64    # reduced
NUM_DOMAINS = 2

SOURCE_DIR = '/content/PlantVillage-Dataset/raw/segmented'
TARGET_DIR = '/content/RiceDiseases-DataSet'

SOURCE_CLASSES = [
    "Peach___Bacterial_spot",
    "Pepper,_bell___Bacterial_spot",
    "Tomato___Bacterial_spot",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Potato___Early_blight",
    "Tomato___Target_Spot",
    "Corn_(maize)___Cercospora_leaf_spot_Gray_leaf_spot",
    "Tomato___Septoria_leaf_spot",
    "Strawberry___Leaf_scorch"
]

TARGET_CLASSES = sorted([d for d in os.listdir(TARGET_DIR) if os.path.isdir(os.path.join(TARGET_DIR, d))])

# ================= GRADIENT REVERSAL =================
@tf.custom_gradient
def grad_reverse(x, lambd):
    def grad(dy):
        return -lambd * dy, None
    return x, grad

class GradientReversal(layers.Layer):
    def __init__(self):
        super().__init__()
        self.lambd = tf.Variable(0.0, trainable=False, dtype=tf.float32)
    def call(self, x):
        return grad_reverse(x, self.lambd)

# ================= AUGMENTATION =================
def strong_aug(x):
    x = tf.image.random_brightness(x, 0.5)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = tf.image.random_flip_left_right(x)
    return tf.clip_by_value(x, 0, 1)

def weak_aug(x):
    x = tf.image.random_brightness(x, 0.3)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    return tf.clip_by_value(x, 0, 1)

def create_views(img, is_source):
    img = tf.cast(img, tf.float32) / 255.0
    aug = strong_aug if is_source else weak_aug
    return aug(img), aug(img)

# ================= DATA GENERATOR =================
def combined_generator():
    datagen = ImageDataGenerator()
    s_gen = datagen.flow_from_directory(
        SOURCE_DIR, classes=SOURCE_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2, shuffle=True, class_mode=None
    )
    t_gen = datagen.flow_from_directory(
        TARGET_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2, shuffle=True, class_mode=None
    )
    while True:
        xs = next(s_gen)
        xt = next(t_gen)
        X = np.concatenate([xs, xt], axis=0)
        domain = tf.concat([tf.zeros(xs.shape[0], dtype=tf.int32),
                            tf.ones(xt.shape[0], dtype=tf.int32)], axis=0)
        v1, v2 = [], []
        for i, img in enumerate(X):
            a, b = create_views(img, i < xs.shape[0])
            v1.append(a)
            v2.append(b)
        yield (tf.stack(v1), tf.stack(v2)), domain

# ================= SE-RESNET BACKBONE =================
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1,1,filters))(se)
    return layers.multiply([x, se])

def residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = se_block(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_encoder():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(16, 3, padding='same')(inp)  # reduced filters
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    for f, s in zip([32, 32, 64, 64, 128, 128], [1,1,2,1,2,1]):  # reduced filters
        x = residual_block(x, f, s)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(FEATURE_DIM, activation='relu', name='shared_features')(x)
    return models.Model(inp, x, name="encoder_agronet")

def projection_head():
    inp = layers.Input((FEATURE_DIM,))
    x = layers.Dense(32, activation='relu')(inp)
    out = layers.Dense(PROJECTION_DIM)(x)
    return models.Model(inp, out, name="projection")

def domain_head(grl):
    inp = layers.Input((FEATURE_DIM,))
    x = grl(inp)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(NUM_DOMAINS, activation='softmax')(x)
    return models.Model(inp, out, name="domain")

# ================= NT-XENT LOSS =================
def nt_xent(z1, z2):
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)
    z = tf.concat([z1, z2], axis=0)
    sim = tf.matmul(z, z, transpose_b=True)
    sim -= tf.eye(tf.shape(z)[0]) * 1e9
    bs = tf.shape(z1)[0]
    labels = tf.concat([tf.range(bs, 2*bs), tf.range(bs)], axis=0)
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, sim / TEMPERATURE, from_logits=True)
    return tf.reduce_mean(loss)

# ================= SIMCLR + MADA-LITE MODEL =================
class SimCLR_DA(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = build_encoder()
        self.projector = projection_head()
        self.grl = GradientReversal()
        self.domain = domain_head(self.grl)

    def train_step(self, data):
        (v1, v2), dom = data
        with tf.GradientTape() as tape:
            h1 = self.encoder(v1, training=True)
            h2 = self.encoder(v2, training=True)
            z1 = self.projector(h1, training=True)
            z2 = self.projector(h2, training=True)
            loss_c = nt_xent(z1, z2)
            pseudo = tf.stop_gradient(tf.nn.softmax(tf.concat([h1,h2], axis=0)))
            weights = tf.reduce_max(pseudo, axis=1)
            dom_pred = self.domain(tf.concat([h1,h2], axis=0), training=True)
            loss_d = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    tf.concat([dom, dom], axis=0), dom_pred
                ) * weights
            )
            loss = loss_c + loss_d
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {"contrastive_loss": loss_c, "domain_loss": loss_d}

# ================= PHASE 1: DOMAIN ADAPTATION =================
model = SimCLR_DA()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4))
gen = combined_generator()
print("\n Training SimCLR + MADA-lite DA")
for epoch in range(EPOCHS_DA):
    model.grl.lambd.assign(min(1.0, epoch / 10.0))
    print(f"Epoch {epoch+1} | DOMAIN_LAMBDA = {model.grl.lambd.numpy():.2f}")
    model.fit(gen, steps_per_epoch=200, epochs=1, verbose=1)
model.encoder.save("encoder_da_agronet.h5")

# ================= PHASE 2: LINEAR EVALUATION =================
model.encoder.trainable = False
classifier = tf.keras.Sequential([
    layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    model.encoder,
    layers.Dense(len(TARGET_CLASSES), activation='softmax')
])
classifier.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_gen = datagen.flow_from_directory(
    TARGET_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32, subset='training', class_mode='categorical'
)
val_gen = datagen.flow_from_directory(
    TARGET_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32, subset='validation', class_mode='categorical', shuffle=False
)

print("\n Training Linear Classifier")
classifier.fit(train_gen, epochs=20, validation_data=val_gen, verbose=1)
loss, acc = classifier.evaluate(val_gen, verbose=0)

# ================= RESULTS =================
classifier.save("full_da_model_agronet.h5")
encoder_size = os.path.getsize("encoder_da_agronet.h5") / 1024  # KB
full_size = os.path.getsize("full_da_model_agronet.h5") / 1024  # KB
print("\n==============================")
print(f"Target Accuracy : {acc*100:.2f}%")
print(f" Encoder Size   : {encoder_size:.2f} KB")
print(f" Full Model Size: {full_size:.2f} KB")
print("==============================")

# ================= TFLITE CONVERSION + INT8 QUANT =================
def representative_data_gen():
    for images, _ in train_gen:
        yield [images.astype(np.float32)]
        break  # only one batch is enough

tflite_model_file = "full_da_model_agronet_int8_small.tflite"
converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_model = converter.convert()
with open(tflite_model_file, "wb") as f:
    f.write(tflite_model)
tflite_size = os.path.getsize(tflite_model_file)  # bytes
print(f" INT8 TFLite Model Size: {tflite_size/1024:.2f} KB")

All 38 classess with adaptation




In [ ]:
# ===============================================================
# SimCLR + MADA-lite + SE-ResNet + INT8 Quantization 38 classes
# ===============================================================

import os, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ================= CONFIG =================
IMG_SIZE = 48
BATCH_SIZE = 32
EPOCHS_DA = 20
TEMPERATURE = 0.1

FEATURE_DIM = 64
PROJECTION_DIM = 64
NUM_DOMAINS = 2

SOURCE_DIR = '/content/PlantVillage-Dataset/raw/segmented'
TARGET_DIR = '/content/RiceDiseases-DataSet'

# AUTO LOAD ALL SOURCE CLASSES
TARGET_CLASSES = sorted([
    d for d in os.listdir(TARGET_DIR)
    if os.path.isdir(os.path.join(TARGET_DIR, d))
])

# ================= GRADIENT REVERSAL =================
@tf.custom_gradient
def grad_reverse(x, lambd):
    def grad(dy):
        return -lambd * dy, None
    return x, grad

class GradientReversal(layers.Layer):
    def __init__(self):
        super().__init__()
        self.lambd = tf.Variable(0.0, trainable=False, dtype=tf.float32)
    def call(self, x):
        return grad_reverse(x, self.lambd)

# ================= AUGMENTATION =================
def strong_aug(x):
    x = tf.image.random_brightness(x, 0.5)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = tf.image.random_flip_left_right(x)
    return tf.clip_by_value(x, 0, 1)

def weak_aug(x):
    x = tf.image.random_brightness(x, 0.3)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    return tf.clip_by_value(x, 0, 1)

def create_views(img, is_source):
    img = tf.cast(img, tf.float32) / 255.0
    aug = strong_aug if is_source else weak_aug
    return aug(img), aug(img)

# ================= DATA GENERATOR =================
def combined_generator():
    datagen = ImageDataGenerator()

    # NO class restriction → loads ALL folders
    s_gen = datagen.flow_from_directory(
        SOURCE_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2,
        shuffle=True,
        class_mode=None
    )

    t_gen = datagen.flow_from_directory(
        TARGET_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2,
        shuffle=True,
        class_mode=None
    )

    while True:
        xs = next(s_gen)
        xt = next(t_gen)

        X = np.concatenate([xs, xt], axis=0)

        domain = tf.concat([
            tf.zeros(xs.shape[0], dtype=tf.int32),
            tf.ones(xt.shape[0], dtype=tf.int32)
        ], axis=0)

        v1, v2 = [], []
        for i, img in enumerate(X):
            a, b = create_views(img, i < xs.shape[0])
            v1.append(a)
            v2.append(b)

        yield (tf.stack(v1), tf.stack(v2)), domain

# ================= SE BLOCK =================
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1,1,filters))(se)
    return layers.multiply([x, se])

# ================= RESIDUAL BLOCK =================
def residual_block(x, filters, stride=1):
    shortcut = x

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = se_block(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(shortcut)

    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

# ================= ENCODER =================
def build_encoder():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    x = layers.Conv2D(16, 3, padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for f, s in zip([32, 32, 64, 64, 128, 128], [1,1,2,1,2,1]):
        x = residual_block(x, f, s)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(FEATURE_DIM, activation='relu', name='shared_features')(x)

    return models.Model(inp, x, name="encoder_agronet")

# ================= HEADS =================
def projection_head():
    inp = layers.Input((FEATURE_DIM,))
    x = layers.Dense(32, activation='relu')(inp)
    out = layers.Dense(PROJECTION_DIM)(x)
    return models.Model(inp, out)

def domain_head(grl):
    inp = layers.Input((FEATURE_DIM,))
    x = grl(inp)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(NUM_DOMAINS, activation='softmax')(x)
    return models.Model(inp, out)

# ================= NT-XENT =================
def nt_xent(z1, z2):
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)

    z = tf.concat([z1, z2], axis=0)
    sim = tf.matmul(z, z, transpose_b=True)
    sim -= tf.eye(tf.shape(z)[0]) * 1e9

    bs = tf.shape(z1)[0]
    labels = tf.concat([tf.range(bs, 2*bs), tf.range(bs)], axis=0)

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels, sim / TEMPERATURE, from_logits=True
    )

    return tf.reduce_mean(loss)

# ================= MODEL =================
class SimCLR_DA(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = build_encoder()
        self.projector = projection_head()
        self.grl = GradientReversal()
        self.domain = domain_head(self.grl)

    def train_step(self, data):
        (v1, v2), dom = data

        with tf.GradientTape() as tape:
            h1 = self.encoder(v1, training=True)
            h2 = self.encoder(v2, training=True)

            z1 = self.projector(h1, training=True)
            z2 = self.projector(h2, training=True)

            loss_c = nt_xent(z1, z2)

            pseudo = tf.stop_gradient(tf.nn.softmax(tf.concat([h1,h2], axis=0)))
            weights = tf.reduce_max(pseudo, axis=1)

            dom_pred = self.domain(tf.concat([h1,h2], axis=0), training=True)

            loss_d = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    tf.concat([dom, dom], axis=0), dom_pred
                ) * weights
            )

            loss = loss_c + loss_d

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        return {"contrastive_loss": loss_c, "domain_loss": loss_d}

# ================= TRAIN DA =================
model = SimCLR_DA()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4))

gen = combined_generator()

print("\n Training DA")
for epoch in range(EPOCHS_DA):
    model.grl.lambd.assign(min(1.0, epoch / 10.0))
    print(f"Epoch {epoch+1} | λ={model.grl.lambd.numpy():.2f}")
    model.fit(gen, steps_per_epoch=200, epochs=1, verbose=1)

model.encoder.save("encoder_da_agronet.h5")

# ================= LINEAR EVAL =================
model.encoder.trainable = False

classifier = tf.keras.Sequential([
    layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    model.encoder,
    layers.Dense(len(TARGET_CLASSES), activation='softmax')
])

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    TARGET_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    subset='training',
    class_mode='categorical'
)

val_gen = datagen.flow_from_directory(
    TARGET_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    subset='validation',
    class_mode='categorical',
    shuffle=False
)

print("\n Training Classifier")
classifier.fit(train_gen, epochs=20, validation_data=val_gen)

loss, acc = classifier.evaluate(val_gen)

# ================= SAVE =================
classifier.save("full_da_model_agronet.h5")

print("\n==============================")
print(f"Target Accuracy : {acc*100:.2f}%")
print("==============================")

# ================= TFLITE =================
def representative_data_gen():
    for images, _ in train_gen:
        yield [images.astype(np.float32)]
        break

converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

with open("model_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("INT8 model ready")